# Day 33: Advanced RAG - Cross-Encoder Reranking

Welcome to Day 33 of your AI Engineering journey!

## Core Theory (Just-in-Time)

**Why Reranking?**
Standard Vector Search (Bi-Encoders) is fast and excellent for initial retrieval across millions of documents. However, it relies on comparing single embeddings independently, missing nuanced relationships between the query and the document.

**The Cross-Encoder Solution**
A Cross-Encoder passes both the user query and the retrieved document simultaneously into a Transformer model. This allows the model's attention mechanism to directly compare the query and document tokens, producing a highly accurate relevance score. Since Cross-Encoders are computationally expensive, we use a two-stage approach:
1.  **Stage 1 (Retrieval):** Use a Bi-Encoder (like standard embeddings in Qdrant) to quickly retrieve the top 10-50 candidates.
2.  **Stage 2 (Reranking):** Pass those top candidates through a Cross-Encoder to re-order them and select the absolute best (e.g., top 3-5) for the LLM.

**Common Production Pitfalls:**
- **Over-Reranking:** Passing hundreds of documents to a Cross-Encoder will cause massive latency spikes. Always keep the initial retrieval pool small (k=10 to 50).
- **Ignoring Score Thresholds:** Cross-Encoders give absolute scores. If the highest score is still very low, your system should probably fallback or say "I don't know" rather than feeding bad context to the LLM.

## Code Implementation

We will implement a two-stage retrieval pipeline using Qdrant as the vector store and LangChain's `ContextualCompressionRetriever` paired with a HuggingFace Cross-Encoder.

### Basic Implementation
Isolating the core concept of cross-encoder reranking with minimal boilerplate.

In [1]:
from typing import List
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

# Initialize embeddings and cross-encoder
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Initialize vector store in memory
client = QdrantClient(":memory:")
collection_name = "basic_reranking"
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# Sample documents (mix of fruit and tech)
docs = [
    "Apples are grown in Washington state and are red or green.",
    "Apple Inc. announced a new iPhone today.",
    "A healthy diet includes fruits like apples and oranges.",
    "The company Apple was founded by Steve Jobs.",
    "To bake an apple pie, you need cinnamon and sugar."
]

# Insert documents into Qdrant
points = []
for doc in docs:
    vec = embeddings.embed_query(doc)
    points.append(PointStruct(id=str(uuid.uuid4()), vector=vec, payload={"page_content": doc}))
client.upsert(collection_name=collection_name, points=points)

# 1. Retrieval (Bi-Encoder)
query = "Who founded the tech company that makes iPhones?"
query_vec = embeddings.embed_query(query)
search_results = client.query_points(collection_name=collection_name, query=query_vec, limit=5)
base_docs = [Document(page_content=pt.payload["page_content"]) for pt in search_results.points]

print("--- Stage 1: Retrieval Results ---")
for i, doc in enumerate(base_docs):
    print(f"{i+1}. {doc.page_content}")

# 2. Reranking (Cross-Encoder)
compressor = CrossEncoderReranker(model=cross_encoder, top_n=3)
reranked_docs = compressor.compress_documents(base_docs, query)

print("\n--- Stage 2: Reranked Results ---")
for i, doc in enumerate(reranked_docs):
    print(f"{i+1}. [Score: {doc.metadata.get('relevance_score', 0.0):.2f}] {doc.page_content}")


/tmp/ipykernel_17893/1962943712.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  35%|███▍      | 69/199 [00:00<00:00, 688.34it/s]

Loading weights:  78%|███████▊  | 155/199 [00:00<00:00, 783.08it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 770.58it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:  70%|██████▉   | 73/105 [00:00<00:00, 717.28it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 773.09it/s]

--- Stage 1: Retrieval Results ---
1. The company Apple was founded by Steve Jobs.
2. Apple Inc. announced a new iPhone today.
3. Apples are grown in Washington state and are red or green.
4. To bake an apple pie, you need cinnamon and sugar.
5. A healthy diet includes fruits like apples and oranges.

--- Stage 2: Reranked Results ---
1. [Score: 0.00] The company Apple was founded by Steve Jobs.
2. [Score: 0.00] Apple Inc. announced a new iPhone today.
3. [Score: 0.00] To bake an apple pie, you need cinnamon and sugar.


### Medium Implementation
Emphasizing clean OOP, state management, and clear object interactions.

In [2]:
class RerankingPipeline:
    def __init__(self, collection_name: str):
        self.collection_name = collection_name
        self.embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
        self.cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
        self.client = QdrantClient(":memory:")
        self._init_db()
        
    def _init_db(self):
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=384, distance=Distance.COSINE)
        )

    def index_documents(self, documents: List[str]):
        points = []
        for doc in documents:
            vec = self.embeddings.embed_query(doc)
            points.append(PointStruct(id=str(uuid.uuid4()), vector=vec, payload={"page_content": doc}))
        self.client.upsert(collection_name=self.collection_name, points=points)

    def retrieve_and_rerank(self, query: str, retrieve_k: int = 5, rerank_k: int = 3) -> List[Document]:
        # Stage 1: Fast Vector Search
        query_vec = self.embeddings.embed_query(query)
        search_results = self.client.query_points(
            collection_name=self.collection_name, 
            query=query_vec, 
            limit=retrieve_k
        )
        base_docs = [Document(page_content=pt.payload["page_content"]) for pt in search_results.points]
        
        # Stage 2: Deep Cross-Encoder Scoring
        compressor = CrossEncoderReranker(model=self.cross_encoder, top_n=rerank_k)
        return compressor.compress_documents(base_docs, query)

# Usage
pipeline = RerankingPipeline("medium_reranking")
pipeline.index_documents(docs)
results = pipeline.retrieve_and_rerank("Which tech giant makes the iPhone?")
print("Medium Implementation Results:")
for doc in results:
    print(f"[{doc.metadata.get('relevance_score', 0.0):.2f}] {doc.page_content}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  39%|███▊      | 77/199 [00:00<00:00, 769.62it/s]

Loading weights:  77%|███████▋  | 154/199 [00:00<00:00, 717.73it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 728.68it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:  61%|██████    | 64/105 [00:00<00:00, 636.50it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 677.83it/s]

Medium Implementation Results:
[0.00] The company Apple was founded by Steve Jobs.
[0.00] Apple Inc. announced a new iPhone today.
[0.00] A healthy diet includes fruits like apples and oranges.


### Advanced Implementation
Production-grade implementation with strict type hinting, docstrings, error handling, AI security (PII checks as an example), and exact import syntax.

In [3]:
import re
import logging
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class RerankerConfig(BaseModel):
    """Configuration for the Reranking Service."""
    collection_name: str = Field(..., description="Name of the Qdrant collection")
    embedding_model: str = Field(default="BAAI/bge-small-en-v1.5")
    cross_encoder_model: str = Field(default="cross-encoder/ms-marco-MiniLM-L-6-v2")
    vector_size: int = Field(default=384)
    similarity_metric: Distance = Field(default=Distance.COSINE)
    min_relevance_score: float = Field(default=0.0, description="Minimum score to return a document")

class SecureRerankingService:
    """
    Production-ready reranking service with AI Security considerations.
    """
    def __init__(self, config: RerankerConfig):
        self.config = config
        self.client = QdrantClient(":memory:")
        
        logger.info(f"Initializing models: {config.embedding_model} and {config.cross_encoder_model}")
        self.embeddings = HuggingFaceEmbeddings(model_name=config.embedding_model)
        self.cross_encoder = HuggingFaceCrossEncoder(model_name=config.cross_encoder_model)
        
        self._setup_collection()
        
    def _setup_collection(self) -> None:
        """Idempotent collection setup."""
        if not self.client.collection_exists(collection_name=self.config.collection_name):
            self.client.create_collection(
                collection_name=self.config.collection_name,
                vectors_config=VectorParams(size=self.config.vector_size, distance=self.config.similarity_metric)
            )
            logger.info(f"Created Qdrant collection: {self.config.collection_name}")

    def _detect_pii(self, text: str) -> bool:
        """Basic PII detection (e.g., catching obvious SSNs or emails) as a security best practice."""
        email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
        if re.search(email_pattern, text):
            return True
        return False

    def index_documents(self, documents: List[str]) -> None:
        """Safely embed and index documents."""
        points = []
        for doc in documents:
            if self._detect_pii(doc):
                logger.warning(f"PII detected in document, skipping indexing: {doc[:30]}...")
                continue
                
            try:
                vec = self.embeddings.embed_query(doc)
                points.append(PointStruct(
                    id=str(uuid.uuid4()), 
                    vector=vec, 
                    payload={"page_content": doc}
                ))
            except Exception as e:
                logger.error(f"Failed to embed document: {e}")
                
        if points:
            self.client.upsert(collection_name=self.config.collection_name, points=points)
            logger.info(f"Successfully indexed {len(points)} documents.")

    def query(self, query_text: str, top_k: int = 5, rerank_k: int = 3) -> List[Document]:
        """
        Two-stage retrieval with fallback mechanisms and strict score thresholding.
        """
        if not query_text.strip():
            logger.warning("Empty query provided.")
            return []

        try:
            # Stage 1
            query_vec = self.embeddings.embed_query(query_text)
            search_results = self.client.query_points(
                collection_name=self.config.collection_name, 
                query=query_vec, 
                limit=top_k
            )
            
            if not search_results.points:
                logger.info("No initial results found.")
                return []

            base_docs = [Document(page_content=pt.payload["page_content"]) for pt in search_results.points]
            
            # Stage 2
            compressor = CrossEncoderReranker(model=self.cross_encoder, top_n=rerank_k)
            reranked_docs = compressor.compress_documents(base_docs, query_text)
            
            # Security / Quality Control: Filter out garbage results
            final_docs = [
                doc for doc in reranked_docs 
                if doc.metadata.get('relevance_score', -999.0) >= self.config.min_relevance_score
            ]
            
            if not final_docs:
                logger.info("All results filtered out by minimum relevance score threshold.")
                # Fallback: Depending on use case, might return top 1 anyway, or empty.
                return []
                
            return final_docs
            
        except Exception as e:
            logger.error(f"Query pipeline failed: {e}")
            # Graceful fallback
            return []

# Usage
config = RerankerConfig(collection_name="advanced_reranking", min_relevance_score=0.1)
service = SecureRerankingService(config)

# Note: One doc has an email (PII) to test the security filter
secure_docs = docs + ["Contact apple-support@apple.com for issues."]
service.index_documents(secure_docs)

results = service.query("Tell me about the founder of Apple.")
print("\nAdvanced Implementation Results:")
for doc in results:
    print(f"[{doc.metadata.get('relevance_score', 0.0):.2f}] {doc.page_content}")


INFO:__main__:Initializing models: BAAI/bge-small-en-v1.5 and cross-encoder/ms-marco-MiniLM-L-6-v2


INFO:sentence_transformers.base.model:No device provided, using cpu


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

INFO:__main__:Successfully indexed 5 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 25.71it/s]


INFO:__main__:All results filtered out by minimum relevance score threshold.



Advanced Implementation Results:


## Practical Lab / Homework

**Task:** Expand the reranking system to handle an edge case: Score Thresholds.

Currently, the `CrossEncoderReranker` always returns the `top_n` documents, even if they are completely irrelevant to the query. 

Your job is to:
1. Add 5 more random documents to the vector store about completely unrelated topics (e.g., cars, space).
2. Create a custom function that intercepts the output of the reranker.
3. Inspect the reranker scores (they are usually stored in `doc.metadata['relevance_score']`).
4. Filter out any documents where the score is below a certain threshold (e.g., 0.0), returning only highly relevant documents.
5. Test it with a query like "How do rockets work?" to ensure it returns nothing instead of returning Apple-related docs.

In [4]:
def search_and_rerank_with_threshold(query: str, threshold: float = 0.0) -> List[Document]:
    """
    Executes a two-stage retrieval process with a strict score threshold.
    """
    query_vec = embeddings.embed_query(query)
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=10
    )
    
    base_docs = [Document(page_content=point.payload["page_content"]) for point in search_result.points]
    
    compressor = CrossEncoderReranker(model=cross_encoder, top_n=5)
    reranked_docs = compressor.compress_documents(base_docs, query)
    
    filtered_docs = [doc for doc in reranked_docs if doc.metadata.get('relevance_score', -999) > threshold]
    
    print(f"\nQuery: '{query}'")
    print(f"--- Filtered Results (Threshold > {threshold}) ---")
    for i, doc in enumerate(filtered_docs):
        print(f"{i+1}. [Score: {doc.metadata.get('relevance_score', 0.0):.2f}] {doc.page_content}")
        
    return filtered_docs

# Insert unrelated documents
unrelated_docs = [
    Document(page_content="The Falcon Heavy rocket launched successfully."),
    Document(page_content="Electric cars are becoming more popular."),
    Document(page_content="SpaceX is a company that builds rockets."),
    Document(page_content="Ford announced a new electric Mustang."),
    Document(page_content="Astronauts travel to the International Space Station.")
]

points = []
for doc in unrelated_docs:
    vec = embeddings.embed_query(doc.page_content)
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vec,
            payload={"page_content": doc.page_content}
        )
    )
client.upsert(collection_name=collection_name, points=points)

search_and_rerank_with_threshold("How do rockets work?")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.83it/s]


Query: 'How do rockets work?'
--- Filtered Results (Threshold > 0.0) ---


[]

## Reference Links
- [LangChain CrossEncoderReranker Documentation](https://python.langchain.com/docs/integrations/document_transformers/cross_encoder_reranker)
- [HuggingFace - Sentence Transformers: Cross-Encoders](https://sbert.net/examples/applications/cross-encoder/README.html)
- [Qdrant - Discovery Search & Reranking](https://qdrant.tech/articles/search-with-discovery-api/)